# Decision Tree 

In [16]:
import numpy as np

## Algorithm

We treat features as nodes. To build a decision tree, we need recursively/iteratively choose the best node for each level. The best node should have the most information gain after spliting. An ideal case would be that if there is a threshold in the feature that can 100% separate all the labels.


### Pesudo code for building a decision tree (training):

- best_feature, best_threshold = `best_split(X, y)`: find out the best feature and threshold to split the dataset, which has the highest information gain
- split the dataset into two subsets based on the best feature and threshold
- recursively build the left and right subtrees using the subsets of data until a stopping criterion is met (e.g., maximum depth, minimum samples per leaf, or pure node)


`best_split(X, y)` is the function to find the best feature and threshold for splitting the dataset. It iterates through all features and possible thresholds, calculates the information gain for each split, and returns the one with the highest gain.
- calculate parent impurity (e.g., entropy or Gini impurity) for the current node
- for each feature:
  - thresholds = unique values of the feature
  - for each threshold in thresholds:
    - split the dataset into left and right subsets based on the threshold
    - calculate the weighted average impurity of the child nodes
    - calculate information gain = parent impurity - weighted average child impurity
- return the feature and threshold with the highest information gain


## Impurity 
The impurity of a node is a measure of how mixed the labels are in that node. The more mixed the labels are, the higher the impurity. The goal of a decision tree is to find splits that reduce the impurity of the nodes.

There are two impurity metrics, Entropy and Gini impurity. 
- Entropy measures the amount of information needed to classify a sample in the node. The more mixed the labels are, the higher the entropy. An ideal split would have an entropy of 0, which means that all the samples in the node belong to the same class.
- Gini impurity measures the probability of a randomly chosen sample being incorrectly classified. An ideal split would have a Gini impurity of 0, which means that all the samples in the node belong to the same class.



### Entropy
Entropy is a way to measure impurity of a node, and an ideal split separates the same labels in the same subtree, which leads to an entropy of 0. The formula for entropy is:

$$H(X) = -\sum_{k=1}^K p_k \log_2 p_k$$

- $X$: dataset or node
- $K$: number of classes
- $p_k$ is the probability of label $k$ in the node, calculated as the fraction of $k$ in the current dataset labels for splitting.
  - For regression problem, this can be represented as variance reduction. 

**Information Gain**
Information gain is the reduction in entropy after a dataset is split on a feature node. The formula for information gain is:

$$\text{IG}(X, split) = H(X) - \sum_{i} w_i H(X_i)$$
- $X$: dataset or node
- $X_i$: child node $i$ after split
- $w_i$: weight of the child node $i$, calculated as the fraction of samples in child node $i$ compared to the parent node



### Gini 

$$G = 1 - \sum_{k=1}^K p_k^2$$

- $p_k$: probability of label $k$ in the node, calculated as the fraction of $k$ in the current dataset labels for splitting.
  
**Gini Reduction**
$$\Delta G(X, split) = G(X) - \sum_{i} w_i G(X_i)$$
- $X$: dataset or node
- $X_i$: child node $i$ after split
- $w_i$: weight of the child node $i$, calculated as the fraction of samples in child node $i$ compared to the parent node

## Implementation using Numpy

In [ ]:
class Node:
    """Node class for a binary decision tree.
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, gain=None):
        self.feature = feature  # index of the feature to split on
        self.threshold = threshold  # threshold value for splitting
        self.left = left  # left child node
        self.right = right  # right child node
        self.value = value  # value for leaf nodes (class label or regression value). Only used for leaf nodes.
        self.gain = gain  # information gain from the split


class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y, depth=0)

    def _stop_building(self, depth, num_labels, num_samples):
        return depth >= self.max_depth or num_labels == 1 or num_samples < self.min_samples_split
    
    def _build_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        num_labels = len(np.unique(y))

        # stopping criteria
        if self._stop_building(depth, num_labels, num_samples):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # find the best split
        best_feature, best_threshold, best_gain = self._best_split(X, y)
        node = Node(feature=best_feature, threshold=best_threshold, gain=best_gain)
        
        # split the dataset
        left_indices = np.where(X[:, best_feature] <= best_threshold)[0]
        right_indices = np.where(X[:, best_feature] > best_threshold)[0]

        # build the left and right subtrees recursively
        node.left = self._build_tree(X[left_indices], y[left_indices], depth + 1)
        node.right = self._build_tree(X[right_indices], y[right_indices], depth + 1)

        return node

    def _best_split(self, X, y, impurity_method='gini'):
        num_samples, num_features = X.shape
        best_gain = -float('inf')
        best_feature, best_threshold = None, None

        parent_impurity = self._calc_impurity(y, method=impurity_method)
        
        # O(dn^2)
        for feature in range(num_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_indices = np.where(X[:, feature] <= threshold)[0]
                right_indices = np.where(X[:, feature] > threshold)[0]
                
                # calculate information gain
                left_impurity = self._calc_impurity(y[left_indices], method=impurity_method)
                right_impurity = self._calc_impurity(y[right_indices], method=impurity_method)
                left_weight = len(left_indices) / num_samples
                right_weight = len(right_indices) / num_samples
                weighted_impurity = left_weight * left_impurity + right_weight * right_impurity
                gain = parent_impurity - weighted_impurity

                # update max gain and best split
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _calc_impurity(self, y, method='gini'):
        if method == 'gini':
            return self._gini(y)
        elif method == 'entropy':
            return self._entropy(y)
        else:
            raise ValueError("Invalid method for impurity calculation. Use 'gini' or 'entropy'.")
    
    def _gini(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        proportions = counts / len(y)
        return 1 - np.sum(proportions ** 2)

    def _entropy(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        proportions = counts / len(y)
        return -np.sum(proportions * np.log2(proportions + 1e-10))  # add small value to avoid log(0)

    
    def _most_common_label(self, y):
        if len(y) == 0:
            return None
        unique_labels, counts = np.unique(y, return_counts=True)
        return unique_labels[np.argmax(counts)]
    
    def predict(self, X):
        X = np.asarray(X)

        # Single sample: shape (n_features,)
        if X.ndim == 1:
            return self._traverse_tree(self.tree, X)

        # Batched samples: shape (n_samples, n_features)
        return np.array([self._traverse_tree(self.tree, x) for x in X])
    
    def _traverse_tree(self, node, X):
        # find a leaf
        if node.value is not None:
            return node.value
        
        # traverse left or right based on threshold
        feature_value = X[node.feature]
        if feature_value <= node.threshold:
            return self._traverse_tree(node.left, X)
        else:
            return self._traverse_tree(node.right, X)
        

## Why the original split search is inefficient
In the original `_best_split`, for each feature you iterate over many candidate thresholds and repeatedly do full-array filtering with `np.where(...)` to build left/right subsets.

If a node has $n$ samples and $d$ features:
- up to $O(n)$ threshold candidates per feature (unique feature values),
- each candidate recomputes masks/indices and impurity from scratch, which costs $O(n)$.

So the per-node cost is roughly $O(d \cdot n^2)$ in the worst case.

## Why sorting + prefix counts is more efficient
The efficient version sorts each feature once, then sweeps the split point from left to right.

For one feature:
1. Sort samples by that feature: $O(n \log n)$.
2. Maintain running class counts for left/right partitions (`left_counts`, `right_counts`).
3. Move one sample at a time across the split and update counts in $O(1)$.
4. Compute impurity from counts (small class dimension), without rebuilding subsets via `np.where`.

This reduces repeated work dramatically:
- per feature: $O(n \log n + n)$ instead of about $O(n^2)$,
- per node: about $O(d \cdot n \log n)$ (plus small class-count overhead).

In short, sorting + prefix-style cumulative counts converts repeated full rescans into one sort plus a linear sweep.

In [ ]:
# Efficient split search using sorting + cumulative class counts
class DecisionTreeEfficient(DecisionTree):
    def _best_split(self, X, y, impurity_method='gini'):
        num_samples, num_features = X.shape
        if num_samples <= 1:
            return None, None, -float('inf')

        # map labels to 0..K-1 so bincount-style logic is safe/fast
        classes, y_encoded = np.unique(y, return_inverse=True)
        num_classes = len(classes)

        # parent impurity from class counts
        _, counts = np.unique(y_encoded, return_counts=True)
        parent_counts = counts.astype(float)
        parent_impurity = self._impurity_from_counts(parent_counts, impurity_method)

        best_gain = -float('inf')
        best_feature = None
        best_threshold = None

        # O(dn log n) due to sorting, but only once per feature
        for feature in range(num_features):
            # sort once per feature
            order = np.argsort(X[:, feature])
            x_sorted = X[order, feature]
            y_sorted = y_encoded[order]

            left_counts = np.zeros(num_classes, dtype=float)
            right_counts = parent_counts.copy()

            # move split point from left to right: after index i
            for i in range(num_samples - 1):
                cls = y_sorted[i]
                left_counts[cls] += 1.0
                right_counts[cls] -= 1.0

                # no valid threshold between identical feature values
                if x_sorted[i] == x_sorted[i + 1]:
                    continue

                n_left = i + 1
                n_right = num_samples - n_left
                if n_left == 0 or n_right == 0:
                    continue

                left_impurity = self._impurity_from_counts(left_counts, impurity_method)
                right_impurity = self._impurity_from_counts(right_counts, impurity_method)

                weighted_impurity = (n_left / num_samples) * left_impurity + (n_right / num_samples) * right_impurity
                gain = parent_impurity - weighted_impurity

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = (x_sorted[i] + x_sorted[i + 1]) / 2.0

        return best_feature, best_threshold, best_gain

    def _impurity_from_counts(self, counts, method='gini'):
        total = counts.sum()
        if total <= 0:
            return 0.0

        p = counts / total

        if method == 'gini':
            return 1.0 - np.sum(p ** 2)
        if method == 'entropy':
            p = p[p > 0]
            return -np.sum(p * np.log2(p))

        raise ValueError("Invalid method for impurity calculation. Use 'gini' or 'entropy'.")

In [19]:
# write a function to print the tree structure
def print_tree(node, depth=0):
    if node is not None:
        if node.value is not None:
            print(f"{' ' * depth * 4}Leaf: value={node.value}")
        else:
            print(f"{' ' * depth * 4}Node: feature={node.feature}, threshold={node.threshold}, gain={node.gain:.4f}")
            print_tree(node.left, depth + 1)
            print_tree(node.right, depth + 1)


In [22]:
# write simple test for the algorithm using the iris dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# load the iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# create and fit the decision tree
tree = DecisionTree(max_depth=3)
tree.fit(X_train, y_train)

# print the tree structure
print("Decision Tree Structure:")
print_tree(tree.tree)

# make predictions and evaluate accuracy
y_pred = tree.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Decision Tree Structure:
Node: feature=2, threshold=1.7, gain=0.3150
    Leaf: value=0
    Node: feature=3, threshold=1.6, gain=0.4102
        Node: feature=2, threshold=4.9, gain=0.0930
            Leaf: value=1
            Leaf: value=2
        Node: feature=2, threshold=4.8, gain=0.0113
            Leaf: value=2
            Leaf: value=2
Test Accuracy: 0.9000
